# 09 — MLP v4: categorical embeddings, shared head

`is_home` / `event` / `result` (and their lags) now go through embedding
tables instead of raw scalar codes — but the output is still one shared
108-wide head. Reconstruction of the checkpoint that preceded the disjoint-heads
version.

In [ ]:
import warnings
from statsbombpy.api_client import NoAuthWarning
from socceraction.data.statsbomb import StatsBombLoader
import pandas as pd
from statsbombpy import sb
import numpy as np
import json
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=NoAuthWarning)
warnings.filterwarnings("ignore", message="credentials were not supplied")

pd.set_option("display.max_columns", None)  # Show all columns

In [ ]:
train = pd.read_csv("data/train_set.csv", index_col=0)
val = pd.read_csv("data/val_set.csv", index_col=0)

## Sentinel remap

Same as the current `mlp_embeddings.ipynb`: `-1` (masked) and `-2`
(no-history) shift into a non-negative space so embeddings and `F.one_hot`
can use them. `target` additionally undoes the `-2->106` convention baked
into the source CSV before applying the same `+2` shift.

In [ ]:
def remap_sentinel(values):
    """Undo the -2->106 convention baked into `target` (if present), then
    shift everything into a non-negative space: -2 -> 0, -1 -> 1, real
    category codes 0.. -> 2.. ."""
    values = np.asarray(values, dtype=float)
    values = np.where(values == 106, -2.0, values)
    return (values + 2).astype(np.int64)

## Dataset construction

`is_home`, `event`, `result` embedded (shared table per field across
current + 3 lags); everything else — spatial/time/duration fields and
match-state fields — fed raw, **unnormalized** (this predates the
normalization work). `target` is one-hot over the full shared 108-wide
vocabulary, for `BCEWithLogits` against a single shared output head.

In [ ]:
continuous_fields = ['start_x', 'start_y', 'time_elapsed', 'duration']
match_state_fields = [
    'period_id', 'minute', 'second',
    'home_goal_count', 'away_goal_count',
    'home_red_count', 'away_red_count',
    'home_yellow_count', 'away_yellow_count',
]
embed_fields = ['is_home', 'event', 'result']


def with_lags(fields, n_lags=3):
    cols = list(fields)
    for lag in range(1, n_lags + 1):
        cols += [f'{f}-{lag}' for f in fields]
    return cols


continuous_cols = with_lags(continuous_fields) + match_state_fields
embed_cols = {f: [f] + [f'{f}-{lag}' for lag in range(1, 4)] for f in embed_fields}

In [ ]:
def build_dataset(df):
    X_cont = torch.tensor(df[continuous_cols].values).float().contiguous()

    X_embed = {}
    for field, cols in embed_cols.items():
        X_embed[field] = torch.tensor(remap_sentinel(df[cols].values)).long().contiguous()

    Y = torch.tensor(remap_sentinel(df['target'].values)).long()
    Y = F.one_hot(Y, num_classes=108).float()

    return X_cont, X_embed, Y

In [ ]:
Xtr_cont, Xtr_embed, Ytr = build_dataset(train)
Xval_cont, Xval_embed, Yval = build_dataset(val)

## Model

Single shared trunk (`W1`-`W3`, 96 hidden) feeding one shared output layer
(`W4`, 96 -> 108) -- no per-field routing, no disjoint heads. Embedding
tables sized from the data (`event`: 26, `is_home`: 4, `result`: 8,
matching the surviving checkpoint's shapes exactly).

In [ ]:
g = torch.Generator().manual_seed(42)

n_hidden = 96
emb_size = 3
vocab_size = 108
input_size = (emb_size * len(Xtr_embed) * 4) + Xtr_cont[0].shape[0]

C = {}
C["event"] = torch.randn((Xtr_embed["event"].max() + 1, emb_size), generator=g)
C["is_home"] = torch.randn((Xtr_embed["is_home"].max() + 1, emb_size), generator=g)
C["result"] = torch.randn((Xtr_embed["result"].max() + 1, emb_size), generator=g)

W1 = torch.randn((input_size, n_hidden), generator=g) * (2 ** 0.5) / (input_size ** 0.5)
b1 = torch.randn(n_hidden, generator=g) * 0
W2 = torch.randn((n_hidden, n_hidden), generator=g) * (2 ** 0.5) / ((W1.shape[1]) ** 0.5)
b2 = torch.randn(n_hidden, generator=g) * 0
W3 = torch.randn((n_hidden, n_hidden), generator=g) * (2 ** 0.5) / ((W2.shape[1]) ** 0.5)
b3 = torch.randn(n_hidden, generator=g) * 0
W4 = torch.randn((n_hidden, vocab_size), generator=g) * 0.001
b4 = torch.randn(vocab_size, generator=g) * 0

parameters = [C["event"], C["is_home"], C["result"], W1, b1, W2, b2, W3, b3, W4, b4]

for p in parameters:
    p.requires_grad = True

lossi = []

## Training

1,000,000 steps, batch 512, validating on the full val set every 100,000
steps (before the weight update, matching the fixed convention established
elsewhere in this project). Single forward path through the shared head --
no per-field masking or routing needed.

In [ ]:
max_steps = 1000000
batch_size = 512
batch_cnt = 100000


def get_lr(step):
    if step < 500000:
        return 0.1
    elif step < 850000:
        return 0.01
    else:
        return 0.001


for i in range(max_steps):
    if i % batch_cnt == 0:
        with torch.no_grad():
            val_embs = []

            for key in Xval_embed:
                emb = C[key][Xval_embed[key]]
                val_embs.append(emb.view(emb.shape[0], -1))

            h = torch.cat([Xval_cont, *val_embs], dim=1)

            for j in range(3, len(parameters), 2):
                hpreact = h @ parameters[j] + parameters[j + 1]

                if j < len(parameters) - 2:
                    h = torch.relu(hpreact)
                else:
                    h = hpreact

            val_loss = F.binary_cross_entropy_with_logits(h, Yval)
            lossi.append(val_loss.item())

    lr = get_lr(i)
    ix = torch.randint(0, Xtr_cont.shape[0], (batch_size,), generator=g)
    Xb_cont, Yb = Xtr_cont[ix], Ytr[ix]

    embs = []

    for key in Xtr_embed:
        emb = C[key][Xtr_embed[key][ix]]
        embs.append(emb.view(emb.shape[0], -1))

    Xb = torch.cat([Xb_cont, *embs], dim=1)

    h = Xb

    for j in range(3, len(parameters), 2):
        hpreact = h @ parameters[j] + parameters[j + 1]

        if j < len(parameters) - 2:
            h = torch.relu(hpreact)
        else:
            h = hpreact

    logits = h

    loss = F.binary_cross_entropy_with_logits(logits, Yb)

    for p in parameters:
        p.grad = None

    loss.backward()

    for p in parameters:
        p.data += -lr * p.grad

In [ ]:
plt.plot(range(len(lossi[1:])), lossi[1:])

In [ ]:
checkpoint = {
    "C1": C["event"],
    "C2": C["is_home"],
    "C3": C["result"],
    "W1": W1,
    "b1": b1,
    "W2": W2,
    "b2": b2,
    "W3": W3,
    "b3": b3,
    "W4": W4,
    "b4": b4,
    "lossi": lossi,
    "max_steps": max_steps,
    "seed": 42,
}
torch.save(checkpoint, "data/mlp_embeddings_checkpoint.pt")